# 1. Carregamento de Dados

- Individuos;
- Domicilios;
- Regionalização;
- CotasPNI;
- AmostraPNI;

In [43]:
import pandas as pd
import numpy as np

In [44]:
dominio_interno = 'luiz.farias'
base = f'C:/Users/{dominio_interno}/Numerator International/BKO - Documents/projeto-dados-ops/do_ams_pni/input'
output = f'C:/Users/{dominio_interno}/Numerator International/BKO - Documents/projeto-dados-ops/do_ams_pni/output'


In [45]:
df_individuos = pd.read_csv(f'{base}/Individuos_vivos.csv', sep=',')
df_domicilios = pd.read_excel(f'C:/Users/luiz.farias/Numerator International/BKO - Documents/projeto-dados-ops/do_bases/TopClient/NRPerfilDomicilio.xls',header=14)
df_cotas_pni42 = pd.read_excel(f'{base}/Cotas_PNI_BR_2025_Agrupado.xlsx',sheet_name='42RegioesPNI')
df_cotas_pni23 = pd.read_excel(f'{base}/Cotas_PNI_BR_2025_Agrupado.xlsx',sheet_name='23RegioesPNI')
df_regiao = pd.read_excel(f'C:/Users/{dominio_interno}/Numerator International/BKO - Documents/projeto-dados-ops/do_gacode/bs_regiões_ihs.xlsx')


C:\Users\luiz.farias\AppData\Local\Temp\ipykernel_20620\3507077369.py:1: DtypeWarning: Columns (0: Telefone_opção_1, 1: Telefone_opção_2, 2: Telefone_opção_3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_individuos = pd.read_csv(f'{base}/Individuos_vivos.csv', sep=',')


In [46]:
df_cotas_pni42.drop(columns=['PAIS','gacodeIdeal'],inplace=True)

In [47]:
df_cotas_pni42.rename(columns={'REGIÃO':'Regioes_42_EXP'},inplace=True)
df_cotas_pni42.drop(columns='REGIAO_LAST',inplace=True)
df_cotas_pni23.rename(columns={'REGIÃO':'Regioes_23_PNI'},inplace=True)


In [48]:
df_individuos.rename(columns={'idDomicilio':'iddomicilio',
                              'painel_2':'FIPanel2'}, inplace=True)


# 2. Transformação de dados

## 2.1 Amostra viva PNI
1. Identificar domicilios que estão no PNC;
2. Identificar domicílios que são um PNI;

In [49]:
# Identificar a amostra viva pertencente ao PNC

# Cruzamento com a base da amostra PNI via left jpjoin.
df_bs_domicilios = df_domicilios.merge(
    df_individuos.loc[df_individuos.FIPanel2.notna()][['iddomicilio', 'FIPanel2']],
    on='iddomicilio',
    how='left')

df_bs_domicilios = df_bs_domicilios[['iddomicilio','FIPanel1','FIPanel2','GAC','NSE', 'Origen Hogar']].copy()

df_bs_domicilios = df_bs_domicilios.merge(df_regiao[['GACODE_KANTAR_Nuevo','Região Expansão 2024','Região Kantar PNI 23 regiones']],
                              left_on='GAC',
                              right_on='GACODE_KANTAR_Nuevo',
                              how='left')

df_bs_domicilios.drop(columns=['GAC'],inplace=True)

df_bs_domicilios.rename(columns={
    'Região Expansão 2024' : 'Regioes_42_EXP',
    'Região Kantar PNI 23 regiones' : 'Regioes_23_PNI'
}, inplace=True)

In [50]:
totais = df_bs_domicilios.FIPanel2.isna().value_counts().reset_index()
totais["FIPanel2"] = totais["FIPanel2"].replace({
    False: "Domicilio com PNI",
    True: "Domicilio sem PNI"
})

totais = totais.rename(columns={"count": "qty"})
totais 


,FIPanel2,qty
0,Domicilio com PNI,24673
1,Domicilio sem PNI,3383


### 2.2 amostra PNI participando

Logica do arquivo de NRDomicilios:
- está vivo no PNC -> Possui data de entrada, mas não possui data de saída;
- está vivo no PNI -> Possui data de entrada, mas não possui data de saída E possui data entrada no PNI E é um domicilio de Origem IBS;

Resultado:
- Amostra Viva PNI

In [51]:
amostra_pni_ibs = df_bs_domicilios.loc[~(df_bs_domicilios.FIPanel2.isna()) & (df_bs_domicilios['Origen Hogar'] == 'IBS')]

In [52]:
'''df_eleg_e2e.Periodo = pd.to_datetime(df_eleg_e2e.Periodo, dayfirst=True, format='%Y-%m')
df_eleg_e2e = df_eleg_e2e.loc[(df_eleg_e2e.Periodo == '2026-07') & (df_eleg_e2e['Elegibilidad Descripcion'] == 'Elegible')][['Periodo','Domicilio','Elegibilidad Descripcion','Actosbuenos','UltimaTransmision']]
df_eleg_e2e['iddomicilio'] = df_eleg_e2e['Domicilio'].astype(int) + 550000000

df_elegiveis_pni_ibs = amostra_pni_ibs.merge(df_eleg_e2e, on='iddomicilio', how='left')'''

"df_eleg_e2e.Periodo = pd.to_datetime(df_eleg_e2e.Periodo, dayfirst=True, format='%Y-%m')\ndf_eleg_e2e = df_eleg_e2e.loc[(df_eleg_e2e.Periodo == '2026-07') & (df_eleg_e2e['Elegibilidad Descripcion'] == 'Elegible')][['Periodo','Domicilio','Elegibilidad Descripcion','Actosbuenos','UltimaTransmision']]\ndf_eleg_e2e['iddomicilio'] = df_eleg_e2e['Domicilio'].astype(int) + 550000000\n\ndf_elegiveis_pni_ibs = amostra_pni_ibs.merge(df_eleg_e2e, on='iddomicilio', how='left')"

## 2.2 Normalização de dados
1. Idade;
2. Identificar dona de casa;
3. NSE, RANGE IDADE, SEXO,

### 2.2.1 aplicação em Domicilios:

In [53]:
df_bs_domicilios['NSE'] = df_bs_domicilios['NSE'].replace({'B1':'AB1', 'A':'AB1'})

In [54]:
import re
import unicodedata

def normalizar_texto(valor):
    if pd.isna(valor):
        return None

    # Converte para string
    valor = str(valor)

    # Remove caracteres invisíveis / controle
    valor = ''.join(
        c for c in valor
        if unicodedata.category(c) not in ('Cc', 'Cf')
    )

    # Normaliza Unicode
    valor = unicodedata.normalize('NFKC', valor)

    # Remove espaços especiais
    valor = valor.replace('\xa0', ' ')
    valor = valor.replace('\u200b', '')
    valor = valor.replace('\ufeff', '')

    # Substitui qualquer sequência de whitespace por um espaço
    valor = re.sub(r'\s+', ' ', valor)

    # Remove espaços nas extremidades
    valor = valor.strip()

    return valor


df_bs_domicilios['Origen Hogar'] = df_bs_domicilios['Origen Hogar'].apply(normalizar_texto)


# ============================================================
# MAPA DE ORIGEM HOGAR → ORIGEM ANTIGA + CÓDIGO DA idOrigem
# ============================================================

map_origem = {
    # -------------------------
    # 42 REGIÕES
    # -------------------------
    'Expansión Nordeste': {
        'origem_antiga': '20 - Expansão Nordeste',
        'idOrigem': 20
    },
    'Expansión Centro-Oeste': {
        'origem_antiga': '22 - Expansión Centro-Oeste',
        'idOrigem': 22
    },
    'Expansión Interior SP': {
        'origem_antiga': '21 - Expansión Interior SP',
        'idOrigem': 21
    },
    'Expansión Resto BR': {
        'origem_antiga': '23 - Expansión Resto BR',
        'idOrigem': 23
    },

    # -------------------------
    # 23 REGIÕES
    # -------------------------
    'IBS': {
        'origem_antiga': '1 - IBS',
        'idOrigem': 1
    },
    'Golondrina': {
        'origem_antiga': '16 - Migrado Golondrina BR',
        'idOrigem': 16
    },
    'Golondrina Nuevo Reclutado BR': {
        'origem_antiga': '18 - Golondrina Nuevo Reclutado BR',
        'idOrigem': 18
    },
    'Mirror Migrado': {
        'origem_antiga': '2 - Mirror Migrado',
        'idOrigem': 2
    },
    'Mirror Nuevo Referido': {
        'origem_antiga': '3 - Mirror Nuevo Referido',
        'idOrigem': 3
    },
    'Mirror Nuevo Reclutado': {
        'origem_antiga': '4 - Mirror Nuevo Reclutado',
        'idOrigem': 4
    }
}

# Criar coluna com a origem no formato antigo
df_bs_domicilios['origem'] = df_bs_domicilios['Origen Hogar'].map(
    lambda x: map_origem.get(x, {}).get('origem_antiga')
)

# Criar coluna contendo somente o código inteiro da idOrigem
df_bs_domicilios['idOrigem'] = df_bs_domicilios['Origen Hogar'].map(
    lambda x: map_origem.get(x, {}).get('idOrigem')
)

In [55]:
df_bs_domicilios.origem.value_counts()

origem
4 - Mirror Nuevo Reclutado            7319
20 - Expansão Nordeste                7123
23 - Expansión Resto BR               2992
21 - Expansión Interior SP            2934
1 - IBS                               2537
22 - Expansión Centro-Oeste           2328
16 - Migrado Golondrina BR            1145
2 - Mirror Migrado                     919
3 - Mirror Nuevo Referido              524
18 - Golondrina Nuevo Reclutado BR     235
Name: count, dtype: int64

In [56]:
df_bs_domicilios['Origen Hogar'].value_counts()

Origen Hogar
Mirror Nuevo Reclutado           7319
Expansión Nordeste               7123
Expansión Resto BR               2992
Expansión Interior SP            2934
IBS                              2537
Expansión Centro-Oeste           2328
Golondrina                       1145
Mirror Migrado                    919
Mirror Nuevo Referido             524
Golondrina Nuevo Reclutado BR     235
Name: count, dtype: int64

### 2.2.2 aplicação em Individuos:

In [57]:
df_indiv = df_individuos[['iddomicilio', 'idIndividuo','Sexo','Idade','DonadeCasa','NSE','FIPanel2']].copy()

In [58]:
map_nse = {
    1: 'AB1',
    2: 'AB1',
    3: 'B2',
    4: 'C1',
    5: 'C2',
    6: 'DE'
}

df_indiv['NSE_norm'] = df_indiv['NSE'].map(map_nse)


df_indiv.Sexo = df_indiv.Sexo.map({'Female': 'Feminino', 'Male': 'Masculino'})

# classificação da variável 'idadeDc' em faixas etárias
bins = [11, 20, 30, 40, 50, np.inf] # Definindo os limites das faixas etárias
labels = ['11-19 anos', '20-29 anos', '30-39 anos','40-49 anos', '50 ou + anos'] # Rótulos para as faixas etárias

df_indiv['faixaEtaria'] = pd.cut(df_indiv['Idade'], bins=bins, labels=labels, right=False) # Abaixo de 11 anos fica NaN (Fora da regra)

# Obter Região do invidivido pela chave de domicílio 
df_indiv = df_indiv.merge(df_bs_domicilios[['iddomicilio','Regioes_42_EXP','Regioes_23_PNI','Origen Hogar','idOrigem']], on='iddomicilio',how='left')



df_indiv.Idade = df_indiv.Idade.fillna(0).astype(int)

In [59]:
df_mono = df_indiv.groupby('iddomicilio')['idIndividuo'].count().reset_index()
df_mono = df_mono.loc[df_mono.idIndividuo == 1]
df_mono.rename(columns={'idIndividuo':'mono_indiv'},inplace=True)
df_indiv = df_indiv.merge(df_mono, on='iddomicilio', how='left')
df_indiv.mono_indiv = df_indiv.mono_indiv.fillna(0)
df_indiv.mono_indiv = df_indiv.mono_indiv.astype(bool)

In [60]:
df_indiv['status_domicilio_gpm'] = df_indiv.iddomicilio.isin(df_bs_domicilios.iddomicilio)
df_indiv = df_indiv.loc[df_indiv.status_domicilio_gpm == True]

In [61]:
df_indiv.DonadeCasa = df_indiv.DonadeCasa.astype(bool)

In [62]:
# Ids que possuem um PNI atrelado ao domicilio
lista_domicilio_pni = df_bs_domicilios.loc[df_bs_domicilios.FIPanel2.notna()].iddomicilio
df_indiv['lar_pni'] = df_indiv.iddomicilio.isin(lista_domicilio_pni)

In [63]:
df_indiv.loc[df_indiv.DonadeCasa == 1].NSE_norm.value_counts()

NSE_norm
AB1    11966
B2      6183
C1      5664
C2      2118
DE      2058
Name: count, dtype: int64

# 3. Matriz agregada da base:

Fluxo:

1. Regioes_42_EXP;
2. Regioes_23_PNI;

Regra de entrada:
- Seleção de individuos leva em conta o desenho amostral das regiões PNI, buscando o individuo que pode compor a amostra segmentada por 23 regiões;
- Atualmente está sendo feito a seleção a partir das Origens de IHS, IBS (1,2,3,4,16,18) com o desenho amostral de 23 regioes e EXP (20,21,22,23) com o desenho de 42 reagiões;

In [64]:
def criar_matriz_amostra(df, fluxo):
    """
    Transforma a tabela de indivíduos em uma matriz agregada por REGIÃO,
    NSE, faixa etária e sexo.

    Espera as colunas:
        REGIÃO
        NSE_norm
        faixaEtaria
        Sexo

    Retorna:
        DataFrame com a estrutura:
        REGIÃO, AB1, B2, C1, C2, DE, TotaisNSE,
        11-19 anos, 20-29 anos, 30-39 anos, 40-49 anos,
        50 ou + anos, TotaisFaixaEtaria,
        Feminino, Masculino, totaisSexo
    """

    df = df.copy()

    # ---------------------------------------------------------
    # Padronização
    # ---------------------------------------------------------
    df['NSE_norm'] = df['NSE_norm'].astype(str).str.strip()
    df['faixaEtaria'] = df['faixaEtaria'].astype(str).str.strip()
    df['Sexo'] = df['Sexo'].astype(str).str.strip()

    # ---------------------------------------------------------
    # NSE
    # ---------------------------------------------------------
    nse = pd.crosstab(
        df[fluxo],
        df['NSE_norm']
    )

    nse = nse.reindex(
        columns=['AB1', 'B2', 'C1', 'C2', 'DE'],
        fill_value=0
    )

    nse['TotaisNSE'] = nse[
        ['AB1', 'B2', 'C1', 'C2', 'DE']
    ].sum(axis=1)

    # ---------------------------------------------------------
    # Faixa etária
    # ---------------------------------------------------------
    idade = pd.crosstab(
        df[fluxo],
        df['faixaEtaria']
    )

    idade = idade.reindex(
        columns=[
            '11-19 anos',
            '20-29 anos',
            '30-39 anos',
            '40-49 anos',
            '50 ou + anos'
        ],
        fill_value=0
    )

    idade['TotaisFaixaEtaria'] = idade[
        [
            '11-19 anos',
            '20-29 anos',
            '30-39 anos',
            '40-49 anos',
            '50 ou + anos'
        ]
    ].sum(axis=1)

    # ---------------------------------------------------------
    # Sexo
    # ---------------------------------------------------------
    sexo = pd.crosstab(
        df[fluxo],
        df['Sexo']
    )

    sexo = sexo.reindex(
        columns=['Feminino', 'Masculino'],
        fill_value=0
    )

    sexo['totaisSexo'] = sexo[
        ['Feminino', 'Masculino']
    ].sum(axis=1)

    # ---------------------------------------------------------
    # Consolidar
    # ---------------------------------------------------------
    resultado = (
        nse
        .join(idade)
        .join(sexo)
        .reset_index()
    )

    # ---------------------------------------------------------
    # Estrutura final
    # ---------------------------------------------------------
    estrutura = [
        fluxo,
        'AB1',
        'B2',
        'C1',
        'C2',
        'DE',
        'TotaisNSE',
        '11-19 anos',
        '20-29 anos',
        '30-39 anos',
        '40-49 anos',
        '50 ou + anos',
        'TotaisFaixaEtaria',
        'Feminino',
        'Masculino',
        'totaisSexo'
    ]

    resultado = resultado.reindex(
        columns=estrutura,
        fill_value=0
    )

    # Garantir valores inteiros
    colunas_numericas = [
        col for col in estrutura
        if col != fluxo
    ]

    resultado[colunas_numericas] = (
        resultado[colunas_numericas]
        .fillna(0)
        .astype(int)
    )

    return resultado

In [65]:
def gap_amostral(ideal, atual, regiao):
    # Merge amsAtualComPni with amsIdealPni to ensure all regions from amsIdealPni are present
    # Use a right merge to keep all rows from amsIdealPni, or a left merge with amsIdealPni as the left
    ideal[regiao] = ideal[regiao].str.strip()
    atual[regiao] = atual[regiao].str.strip()
    
    merged_df = pd.merge(ideal, atual, on=[regiao], how='left', suffixes=('_ideal', '_atual'))

    # Identify the numeric columns from the original amsIdealPni DataFrame
    numeric_cols = ideal.select_dtypes(include=np.number).columns.tolist()

    # Fill NaN values in the columns that came from amsAtualComPni with 0
    for col in numeric_cols:
        merged_df[f'{col}_atual'] = merged_df[f'{col}_atual'].fillna(0)

    # Calculate the saldo for numeric columns by subtracting the '_ideal' columns from the '_atual' columns
    Saldo_numerico =  merged_df[[f'{col}_atual' for col in numeric_cols]].values - merged_df[[f'{col}_ideal' for col in numeric_cols]].values

    # Create a new DataFrame for the saldo with the original column names and include 'PAIS' and 'gacodeIdeal'
    # Select the non-numeric columns directly from merged_df where they originated from amsIdealPni
    Saldo = merged_df[[regiao]].copy()
    Saldo[numeric_cols] = Saldo_numerico.astype(int) # Ensure numeric columns are integer type


    # Ensure column order is consistent with amsIdealPni (or the desired order)
    # The columns are already in the desired order after selecting them explicitly
    # Saldo.rename(columns={'PAIS_ideal': 'PAIS', 'gacodeIdeal_ideal': 'gacodeIdeal'}, inplace=True) # Not needed anymore
    Saldo = Saldo[[regiao] + numeric_cols]

    saldoComPni = Saldo
    return saldoComPni

In [66]:
df_bs_domicilios.columns

Index(['iddomicilio', 'FIPanel1', 'FIPanel2', 'NSE', 'Origen Hogar',
       'GACODE_KANTAR_Nuevo', 'Regioes_42_EXP', 'Regioes_23_PNI', 'origem',
       'idOrigem'],
      dtype='str')

In [67]:
# Amostra PNI atual:
# PNI not null
# Deve existir apenas 1 PNI por Domicilio
# Há dois fluxos, um para as origens de Expansão e outro para IHS (enteno que deveria ser para as mesmmas regiões de PNI)

# Matrizes
matriz_amostra_pni_atual = criar_matriz_amostra(
    df_indiv.loc[(df_indiv.FIPanel2.notna()) & 
                 (df_indiv.idOrigem.isin([1,2,3,4,18,16]))],
    fluxo='Regioes_23_PNI')

matriz_amostra_exp_atual = criar_matriz_amostra(
    df_indiv.loc[(df_indiv.FIPanel2.notna()) &
                 (df_indiv.idOrigem.isin([20,21,22,23]))],
    fluxo='Regioes_42_EXP')

# Bases de Domicilio
bs_dom_universo_23_pni = df_bs_domicilios.loc[
    (df_bs_domicilios.FIPanel2.isna()) &
    (df_bs_domicilios.idOrigem.isin([1,2,3,4,18,16]))&
    (df_bs_domicilios.NSE.notna())& (df_bs_domicilios.Regioes_42_EXP.notna())]

bs_dom_universo_42_exp = df_bs_domicilios.loc[
    (df_bs_domicilios.FIPanel2.isna()) &
    (df_bs_domicilios.idOrigem.isin([20,21,22,23]))&
    (df_bs_domicilios.NSE.notna())& (df_bs_domicilios.Regioes_42_EXP.notna())]

# Bases de individuo
bs_indiv_universo_23_pni = df_indiv.loc[
    (df_indiv.FIPanel2.isna()) & 
    (df_indiv.idOrigem.isin([1,2,3,4,18,16]))&
    (df_indiv.NSE.notna()) & (df_indiv.Regioes_42_EXP.notna()) & (df_indiv.Sexo.notna())]

bs_indiv_universo_42_exp = df_indiv.loc[
    (df_indiv.FIPanel2.isna()) & 
    (df_indiv.idOrigem.isin([20,21,22,23])) &
    (df_indiv.NSE.notna()) & (df_indiv.Regioes_42_EXP.notna()) & (df_indiv.Sexo.notna())]


matriz_amostra_exp_atual = matriz_amostra_exp_atual.sort_values(by='Regioes_42_EXP')
matriz_amostra_pni_atual = matriz_amostra_pni_atual.sort_values(by='Regioes_23_PNI')

df_cotas_pni23 = df_cotas_pni23.sort_values(by='Regioes_23_PNI')
df_cotas_pni42 = df_cotas_pni42.sort_values(by='Regioes_42_EXP')



In [68]:
gap_amostral_23_regioes = gap_amostral(df_cotas_pni23, matriz_amostra_pni_atual, 'Regioes_23_PNI')
gap_amostral_42_regioes = gap_amostral(df_cotas_pni42, matriz_amostra_exp_atual, 'Regioes_42_EXP')


# 4. Universo de candidatos reais para participação do PNI:

Regras para indivíduo apto:

- Domicilio do indivíduo não deve possuir um PNI (já definido);
- Não deve ser menor de 11 anos;
- Não deve ser a dona de casa, a menos que seja o unico indiv

In [69]:
def candidatos_reais(df_indiv_filtrado_origem, df_bs_domicilios_filtrado_origem):    
    # Parametros:
        # Dataframe dos individuos deve estar filtrado por origem para a divisão do fluxo em duas partes o filtro de origem e do painel PNI;
        # DataFrame dos domicilios não PNI já aplicado e o filtro de origem e do painel PNI;
    # Retorno:
        # Data frame com os candidator reais dentro do universo selecionado:


    # 1. Seleção do universo de individuos disponíveis para compor o painel PNI
        # Os candidatos são os individuos que não possuem um pni no lar
    df_candidatos = df_indiv_filtrado_origem.loc[
        df_indiv_filtrado_origem.lar_pni == False     
    ]

    # 2. Criação de flags de identificação
    df_candidatos['flag_candidato'] = np.select(
        [   
            (df_candidatos.Idade >= 11) & (df_candidatos.DonadeCasa == False),
            (df_candidatos.Idade >= 11) & (df_candidatos.DonadeCasa == True) & (df_candidatos.mono_indiv == True ), 
            (df_candidatos.Idade < 11),        

        ],
        [
            'Individuo do lar > 11 anos',
            'Dona de casa do lar > 11 anos E mono_indi',
            'individuo do lar < 11 anos'
        ],
        default= 'Exclusão por regra'

    )

    # 3. Definição do universo de domicilios que precisam de 1 PNI
    df_bs_universo_domicilios_not_pni = df_bs_domicilios_filtrado_origem

    # 4. Identificação dos candidatos reais por domicilio
    flt = [
        'Dona de casa do lar > 11 anos E mono_indi',
        'Individuo do lar > 11 anos'
    ]
    candidatos_regras = df_candidatos.loc[df_candidatos.flag_candidato.isin(flt)]

    df_candidatos_reais = candidatos_regras.groupby('iddomicilio')['idIndividuo'].count().reset_index()
    df_candidatos_reais.rename(columns={'idIndividuo':'candidatos_reais'},inplace=True)

    df_bs_universo_domicilios_not_pni = df_bs_universo_domicilios_not_pni.merge(df_candidatos_reais,on='iddomicilio', how='left')

    # 5. Separação dos numero de candidator por faixas
    df_bs_universo_domicilios_not_pni['candidatos_reais'] = df_bs_universo_domicilios_not_pni['candidatos_reais'].fillna(0)
    
    bins = [-1, 0,1, 3, 6, float('inf')]
    labels = ['0 candidatos','1 candidato','2-3 candidatos','4-6 candidatos','7+ candidatos'
    ]   
    
    df_bs_universo_domicilios_not_pni['faixa_candidatos'] = pd.cut(
            df_bs_universo_domicilios_not_pni['candidatos_reais'],
            bins=bins,
            labels=labels,
            include_lowest=True
    )

    return df_bs_universo_domicilios_not_pni, candidatos_regras

In [70]:
df_dom_universo_selecao_23_pni,df_indiv_candidatos_selecao_23_pni = candidatos_reais(bs_indiv_universo_23_pni,bs_dom_universo_23_pni)
df_dom_universo_selecao_42_exp,df_indiv_candidatos_selecao_42_exp = candidatos_reais(bs_indiv_universo_42_exp,bs_dom_universo_42_exp)

In [71]:
ordem = [
    '0 candidatos',
    '1 candidato',
    '2-3 candidatos',
    '4-6 candidatos',
    '7+ candidatos'
]

df_dom_universo_selecao_42_exp['faixa_candidatos'] = pd.Categorical(
    df_dom_universo_selecao_42_exp['faixa_candidatos'],
    categories=ordem,
    ordered=True
)

df_perc_universo_42_exp = (
    df_dom_universo_selecao_42_exp['faixa_candidatos']
    .value_counts(dropna=False, sort=False)
    .to_frame('qtd')
    .assign(
        proporcao=lambda x: x['qtd'] / x['qtd'].sum() * 100
    )
    .reset_index()
)

df_perc_universo_23_pni = (
    df_dom_universo_selecao_23_pni['faixa_candidatos']
    .value_counts(dropna=False, sort=False)
    .to_frame('qtd')
    .assign(
        proporcao=lambda x: x['qtd'] / x['qtd'].sum() * 100
    )
    .reset_index()
)



In [72]:
total_geral = bs_dom_universo_42_exp[bs_dom_universo_42_exp.FIPanel2.isna()].iddomicilio.nunique() + bs_dom_universo_23_pni[bs_dom_universo_23_pni.FIPanel2.isna()].iddomicilio.nunique()
total_painel_PNI = bs_dom_universo_23_pni[bs_dom_universo_23_pni.FIPanel2.isna()].iddomicilio.nunique()
total_painel_EXP = bs_dom_universo_42_exp[bs_dom_universo_42_exp.FIPanel2.isna()].iddomicilio.nunique()

print('Total de indivíduos não participantes do PNI: ' , total_geral)
print('Total de candidatos do Expansão (sem aplicação da regra): ' , total_painel_EXP)
print('Total de candidatos do PNI (sem aplicação da regra): ' , total_painel_PNI)

print('\nProporção de candidados por Universo (42 Expansão): \n')
display(df_perc_universo_42_exp)

print('\nProporção de candidados por Universo (23 PNI): \n')
display(df_perc_universo_23_pni)


Total de indivíduos não participantes do PNI:  3315
Total de candidatos do Expansão (sem aplicação da regra):  1930
Total de candidatos do PNI (sem aplicação da regra):  1385

Proporção de candidados por Universo (42 Expansão): 



,faixa_candidatos,qtd,proporcao
0,0 candidatos,270,13.989637
1,1 candidato,773,40.051813
2,2-3 candidatos,715,37.046632
3,4-6 candidatos,161,8.341969
4,7+ candidatos,11,0.569948



Proporção de candidados por Universo (23 PNI): 



,faixa_candidatos,qtd,proporcao
0,0 candidatos,113,8.158845
1,1 candidato,626,45.198556
2,2-3 candidatos,516,37.256318
3,4-6 candidatos,121,8.736462
4,7+ candidatos,9,0.649819


# 5. Seleção de indivíduos

In [73]:
def selecao(gap_amostral,df_dom_universo_selecao, df_indiv_candidatos_selecao, regiao):
    df_dom_universo_selecao[regiao] = df_dom_universo_selecao[regiao].str.strip()
    df_indiv_candidatos_selecao[regiao] = df_indiv_candidatos_selecao[regiao].str.strip()
    gap_amostral[regiao] = gap_amostral[regiao].str.strip()

    df_dom_universo_selecao = df_dom_universo_selecao.loc[df_dom_universo_selecao.NSE.notna()]
    df_indiv_candidatos_selecao = df_indiv_candidatos_selecao.loc[df_indiv_candidatos_selecao.NSE.notna()]
    
    dom_1_candidato_23_pni = df_dom_universo_selecao.loc[
    df_dom_universo_selecao['faixa_candidatos'] == '1 candidato',
    'iddomicilio'
    ]

    df_selecao_direta_23_exp = df_indiv_candidatos_selecao.loc[
        df_indiv_candidatos_selecao['iddomicilio'].isin(dom_1_candidato_23_pni)
    ].copy()

    dom_multiplos = df_dom_universo_selecao.loc[
        df_dom_universo_selecao['faixa_candidatos'].isin([
            '2-3 candidatos',
            '4-6 candidatos',
            '7+ candidatos'
        ]),
        'iddomicilio'
    ]

    df_candidatos_decisao = df_indiv_candidatos_selecao.loc[
        df_indiv_candidatos_selecao['iddomicilio'].isin(dom_multiplos)
    ].copy()

    df_candidatos_decisao_23_pni_decisao = df_indiv_candidatos_selecao.copy()

    gap_nse = gap_amostral.set_index(regiao)

    gap_nse = gap_amostral.set_index(regiao)

    df_candidatos_decisao['gap_nse'] = [
        gap_nse.loc[regiao, nse]
        for regiao, nse in zip(
            df_candidatos_decisao[regiao],
            df_candidatos_decisao['NSE_norm']
        )
    ]

    df_candidatos_decisao['gap_faixa'] = [
        gap_nse.loc[regiao, faixa]
        for regiao, faixa in zip(
            df_candidatos_decisao[regiao],
            df_candidatos_decisao['faixaEtaria']
        )
    ]

    df_candidatos_decisao['gap_sexo'] = [
        gap_nse.loc[regiao, sexo]
        for regiao, sexo in zip(
            df_candidatos_decisao[regiao],
            df_candidatos_decisao['Sexo']
        )
    ]

    # pontuação
    df_candidatos_decisao['score_nse'] = -df_candidatos_decisao['gap_nse'].clip(upper=0)
    df_candidatos_decisao['score_faixa'] = -df_candidatos_decisao['gap_faixa'].clip(upper=0)
    df_candidatos_decisao['score_sexo'] = -df_candidatos_decisao['gap_sexo'].clip(upper=0)

    df_candidatos_decisao['score'] = (
        df_candidatos_decisao['score_nse']
        + df_candidatos_decisao['score_faixa']
        + df_candidatos_decisao['score_sexo']
    )

    df_candidatos_decisao['selecionado'] = 0

    idx = (
        df_candidatos_decisao
        .groupby('iddomicilio')['score']
        .idxmax()
    )

    df_candidatos_decisao.loc[idx, 'selecionado'] = 1

    df_indiv_candidatos_selecao['tipo_selecao'] = 'Não selecionado'

    df_indiv_candidatos_selecao.loc[
        df_indiv_candidatos_selecao['iddomicilio'].isin(dom_1_candidato_23_pni),
        'tipo_selecao'
    ] = 'Direto'

    df_indiv_candidatos_selecao.loc[
        df_indiv_candidatos_selecao.index.isin(
            df_candidatos_decisao.loc[
                df_candidatos_decisao['selecionado'] == 1
            ].index
        ),
        'tipo_selecao'
    ] = 'Score'


    df_selecionados_score = df_candidatos_decisao.loc[
        df_candidatos_decisao['selecionado'] == 1
    ].copy()

    df_selecionados_direto = df_indiv_candidatos_selecao.loc[
        df_indiv_candidatos_selecao['tipo_selecao'] == 'Direto'
    ].copy()

    df_selecao_final = pd.concat(
        [
            df_selecionados_direto,
            df_selecionados_score
        ],
        ignore_index=True
    )

    print('Selecionados diretos:', len(df_selecionados_direto))
    print('Selecionados por score:', len(df_selecionados_score))
    print('Total selecionado:', len(df_selecao_final))
    print('Domicílios distintos:', df_selecao_final['iddomicilio'].nunique())

    return df_selecao_final

In [74]:
df_selecionados_23_pni = selecao(gap_amostral_23_regioes,df_dom_universo_selecao_23_pni, df_indiv_candidatos_selecao_23_pni, 'Regioes_23_PNI')
    

Selecionados diretos: 626
Selecionados por score: 646
Total selecionado: 1272
Domicílios distintos: 1272


In [75]:
df_selecionados_42_exp = selecao(gap_amostral_42_regioes,df_dom_universo_selecao_42_exp, df_indiv_candidatos_selecao_42_exp, 'Regioes_42_EXP')


Selecionados diretos: 773
Selecionados por score: 887
Total selecionado: 1660
Domicílios distintos: 1660


In [76]:
df_selecionados_42_exp.loc[df_selecionados_42_exp.tipo_selecao == 'Direto', 'selecionado'] = 1
df_selecionados_23_pni.loc[df_selecionados_23_pni.tipo_selecao == 'Direto', 'selecionado'] = 1

df_selecionados_42_exp.loc[df_selecionados_42_exp.score.notna(), 'tipo_selecao'] = 'Score'
df_selecionados_23_pni.loc[df_selecionados_23_pni.score.notna(), 'tipo_selecao'] = 'Score'

In [77]:
df_selecionados_42_exp = df_selecionados_42_exp[[
       'iddomicilio', 'idIndividuo', 'Regioes_42_EXP', 'idOrigem',
       'status_domicilio_gpm','flag_candidato', 'tipo_selecao']]

In [78]:
df_selecionados_23_pni = df_selecionados_23_pni[[
       'iddomicilio', 'idIndividuo', 'Regioes_23_PNI', 'idOrigem',
       'status_domicilio_gpm','flag_candidato', 'tipo_selecao']]

In [79]:
df_lista_importacao = pd.concat([df_selecionados_23_pni[['idIndividuo']], df_selecionados_42_exp[['idIndividuo']]])
df_lista_importacao['painel'] = 2

In [80]:
from datetime import datetime

data = datetime.now().strftime("%d%m%Y")

# DataFrames que você já possui
# df1, df2, df3, df4

nm = f'/PROD_SELECAO_PNI_{data}.xlsx'

with pd.ExcelWriter(output + nm, engine="openpyxl") as writer:
    df_lista_importacao.to_excel(writer, sheet_name="Lista Para Importação", index=False)
    df_selecionados_23_pni.to_excel(writer, sheet_name="Detalhe 23 PNI", index=False)
    df_selecionados_42_exp.to_excel(writer, sheet_name="Detalhe 42 EXP", index=False)
    gap_amostral_23_regioes.to_excel(writer, sheet_name="Gap Amostral 23 PNI", index=False)
    gap_amostral_42_regioes.to_excel(writer, sheet_name="Gap Amostral 42 EXP", index=False)


print(f"Arquivo gerado com sucesso: {output + nm}")

Arquivo gerado com sucesso: C:/Users/luiz.farias/Numerator International/BKO - Documents/projeto-dados-ops/do_ams_pni/output/PROD_SELECAO_PNI_16092026.xlsx
